## Récupération des données Wikidata et enrichissement des fichiers XML et CSV d'index pour les lieux

On cherche ici à récupérer les informations relatives à chaque lieu à partir de l'URL indiquée dans l'attribut @source de la balise <place>.
Pour chaque lieu, on part de 
```<place xml:id="NomLieu" source="URL"></place>``` pour arriver à 

```
<place xml:id="NomLieu" source="URL">
        <placeName xml:lang="fr">Nom en français</placeName>
        <placeName xml:lang="it">Nom en italien</placeName>
        <location>
          <region type="continent">Nom du continent</region>
          <country>Nom du pays</country>
          <geo>coordonnées géographiques</geo>
        </location>
        <idno type="wikidata">Identifiant Wikidata</idno>
      </place>
```

Ensuite, les entrées ajoutées doivent être classées par continent puis pays puis par ordre alphabétique des noms parmi les entrées existantes,. 
Enfin, on met à jour le fichier CSV. 
      

#### Import des librairies nécessaires

In [ ]:
import xml.etree.ElementTree as ET
import requests
from time import sleep
import csv

#### Espace de nom TEI

In [ ]:
ns = {"tei": "http://www.tei-c.org/ns/1.0"}
ET.register_namespace("", ns["tei"])

#### Chemin des fichiers XML et CSV

In [ ]:
INPUT_XML = "../../IndexArterm/IndexLieuxComplet.xml"
OUTPUT_CSV = "IndexLieux.csv"

In [ ]:
# HEADERS

HEADERS_WIKIDATA = {
            "User-Agent": "MyWikidataBot/1.0 (contact: pierrehusson482@gmail.com)"
        }

#### Définition des fonctions utiles

In [ ]:
# Fonction pour extraire le QID d'une URL Wikidata
def extract_qid(url):
    return url.strip().split("/")[-1] if url and "wikidata.org" in url else None

# Fonction pour récupérer les informations d'un lieu à partir de Wikidata
def get_label(qid, lang="fr"):
    try:
        r = requests.get(
            f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json",
            headers=HEADERS_WIKIDATA
        )
        return r.json()["entities"][qid]["labels"].get(lang, {}).get("value", "")
    except:
        return ""

# Fonction pour récupérer les labels dans plusieurs langues, ici le français et l'italien
# (peut être étendu à d'autres langues si nécessaire)
def get_labels(qid, langs=["fr", "it"]):
    try:
        r = requests.get(
            f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json",
            headers=HEADERS_WIKIDATA
        )
        data = r.json()["entities"][qid]
        return {lang: data["labels"].get(lang, {}).get("value", "") for lang in langs}
    except:
        return {}

# Fonction pour enrichir les données d'un lieu à partir de Wikidata
def get_wikidata_place_data(qid):
    try:
        r = requests.get(
            f"https://www.wikidata.org/wiki/Special:EntityData/{qid}.json",
            headers=HEADERS_WIKIDATA
        )

        r.raise_for_status()  # <-- pour déclencher une exception en cas d’erreur HTTP
        data = r.json()["entities"][qid]
        claims = data.get("claims", {})

        # Coordonnées
        coords = claims.get("P625", [{}])[0].get("mainsnak", {}).get("datavalue", {}).get("value", {})
        lat = coords.get("latitude")
        lon = coords.get("longitude")
        geo = f"{lat} {lon}" if lat and lon else ""

        # Pays
        country = ""
        if "P17" in claims:
            country_qid = claims["P17"][0]["mainsnak"]["datavalue"]["value"]["id"]
            country = get_label(country_qid)

        # Continent
        continent = ""
        if "P30" in claims:
            cont_qid = claims["P30"][0]["mainsnak"]["datavalue"]["value"]["id"]
            continent = get_label(cont_qid)

        # Noms
        labels = get_labels(qid)

        return {
            "labels": labels,
            "geo": geo,
            "country": country,
            "continent": continent
        }

    except Exception as e:
        print(r.status_code, r.url)
        print(r.text[:500])
        print(f"❌ Erreur Wikidata {qid} : {e}")
        return None

def is_enriched(place):
    return place.find("tei:location", ns) is not None or place.find("tei:placeName", ns) is not None

#### Charger et parser le fichier XML

In [ ]:
tree = ET.parse(INPUT_XML)
root = tree.getroot()
listPlace = root.find(".//tei:listPlace", ns)
places = listPlace.findall("tei:place", ns)

#### Enrichissement du fichier XML

In [ ]:
records = []

for place in places:
    xml_id = place.get("{http://www.w3.org/XML/1998/namespace}id")
    source = place.get("source", "")
    qid = extract_qid(source)

    name_fr = name_it = continent = country = geo = ""

    if qid and not is_enriched(place):
        data = get_wikidata_place_data(qid)
        if not data:
            continue
        labels = data["labels"]
        name_fr = labels.get("fr", "")
        name_it = labels.get("it", "")
        continent = data["continent"]
        country = data["country"]
        geo = data["geo"]

        # Suppression du contenu existant
        for child in list(place):
            place.remove(child)

        # placeName en plusieurs langues
        for lang, label in labels.items():
            if label:
                pn = ET.SubElement(place, f"{{{ns['tei']}}}placeName")
                pn.set("{http://www.w3.org/XML/1998/namespace}lang", lang)
                pn.text = label

        # <location>
        loc = ET.SubElement(place, f"{{{ns['tei']}}}location")
        if continent:
            ET.SubElement(loc, f"{{{ns['tei']}}}region", {"type": "continent"}).text = continent
        if country:
            ET.SubElement(loc, f"{{{ns['tei']}}}country").text = country
        if geo:
            ET.SubElement(loc, f"{{{ns['tei']}}}geo").text = geo

        # idno wikidata
        ET.SubElement(place, f"{{{ns['tei']}}}idno", {"type": "wikidata"}).text = qid
        sleep(0.5)

    # conserver les entrées déjà enrichies
    else:
        for pn in place.findall("tei:placeName", ns):
            lang = pn.attrib.get("{http://www.w3.org/XML/1998/namespace}lang", "")
            if lang == "fr":
                name_fr = pn.text
            elif lang == "it":
                name_it = pn.text

        loc = place.find("tei:location", ns)
        if loc is not None:
            reg = loc.find("tei:region", ns)
            if reg is not None and reg.attrib.get("type") == "continent":
                continent = reg.text
            country_el = loc.find("tei:country", ns)
            if country_el is not None:
                country = country_el.text
            geo_el = loc.find("tei:geo", ns)
            if geo_el is not None:
                geo = geo_el.text

        wikidata_id = place.find("tei:idno[@type='wikidata']", ns)
        if wikidata_id is not None:
            qid = wikidata_id.text

    records.append({
        "xml:id": xml_id or "",
        "name_fr": name_fr or "",
        "name_it": name_it or "",
        "continent": continent or "",
        "country": country or "",
        "geo": geo or "",
        "wikidata": qid or ""
    })

#### Tri par ordre alphabétique et sauvegarde du fichier XML trié

In [ ]:
# === TRI PAR continent > pays > nom ===
records.sort(key=lambda r: (r["continent"], r["country"], r["name_fr"]))

# === RE-ORDONNANCER LES <place> ===
for p in places:
    listPlace.remove(p)
for record in records:
    place = next((p for p in places if p.get("{http://www.w3.org/XML/1998/namespace}id") == record["xml:id"]), None)
    if place is not None:
        listPlace.append(place)

tree.write(INPUT_XML, encoding="utf-8", xml_declaration=True)

#### Création du fichier CSV à partir du fichier XML

In [ ]:
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=["xml:id", "name_fr", "name_it", "continent", "country", "geo", "wikidata"])
    writer.writeheader()
    for row in records:
        writer.writerow(row)

print(f"✅ XML mis à jour : {INPUT_XML}")
print(f"✅ CSV généré     : {OUTPUT_CSV}")
